# Notebook 05 — Delay Root-Cause Classifier (InLegalBERT, Zero-Shot)

**Objective (from your seminar):** "Build a delay root-cause classifier using pretrained legal NLP techniques."

**What this notebook does, in plain terms:** For each case, we have raw text (`facts`, `judgment_reason`). We want to figure out the most likely REASON a case is delayed — without manually labeling hundreds of examples first. We do this using **InLegalBERT** (a BERT model pretrained specifically on Indian legal text) in a **zero-shot** way: instead of training the model on our own labeled examples, we describe each possible delay-cause category in a sentence, convert both the case text and the category descriptions into numeric vectors ("embeddings"), and pick whichever category is closest in meaning to each case.

---
### ⚠️ Important note before running
This sandbox has no `transformers`/`torch` installed and no internet, so — same as the Cox model notebook — I could not execute or test this code here. It is written carefully and should run correctly on your machine. Before running:
```
pip install transformers torch
```
The first time you run Step 5 (loading InLegalBERT), it will **download the model automatically** (a few hundred MB) — make sure you have internet for that first run. After that, it's cached locally.

In [1]:
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data" / "processed"
MODELS = ROOT / "models"
DATA.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)
print("Project root:", ROOT)


Project root: e:\Projects\Judicial AI system with drift detection\Notebook


## Step 1 — Import required libraries

In [2]:
import torch
print(torch.__version__)

2.14.0+cpu


In [3]:
# transformers -- Hugging Face library that lets us load pretrained models like InLegalBERT
from transformers import AutoTokenizer, AutoModel

# torch -- the deep learning framework InLegalBERT runs on
import torch

# pandas -- for loading and working with our dataset
import pandas as pd

# cosine_similarity -- measures how "close" two pieces of text are in meaning,
# once they've been converted into numeric vectors (embeddings)
from sklearn.metrics.pairwise import cosine_similarity

# numpy -- for array/vector math
import numpy as np

print("Libraries imported successfully")


e:\Projects\Judicial AI system with drift detection\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries imported successfully


## Step 2 — Load the structured dataset (from Notebook 03)

In [4]:
# Load the structured dataset we built in Notebook 03
# (this has the case text columns: facts, judgment_reason, etc.)
df = pd.read_csv(DATA / "structured_dataset.csv")

print("Loaded shape:", df.shape)
df[['case_id', 'crime_type', 'facts', 'judgment_reason']].head(3)


FileNotFoundError: [Errno 2] No such file or directory: 'e:\\Projects\\Judicial AI system with drift detection\\Notebook\\data\\processed\\structured_dataset.csv'

## Step 3 — Combine case text into a single field
The model needs one block of text per case to analyze.

In [ ]:
# The model needs ONE text block per case to analyze, not two separate columns.
# We combine 'facts' and 'judgment_reason' into a single text field, since
# together they usually contain the clues about WHY a case is taking long
# (e.g. mentions of pending investigation, witness issues, adjournments).

df['case_text'] = (
    df['facts'].fillna('') + ". " + df['judgment_reason'].fillna('')
)

# Quick look at how long these combined texts are (helps us know if we need
# to truncate long texts later -- InLegalBERT has a max input length limit)
df['case_text_length'] = df['case_text'].str.len()
print(df['case_text_length'].describe())


count    365.000000
mean     587.715068
std      120.887451
min      336.000000
25%      493.000000
50%      594.000000
75%      676.000000
max      994.000000
Name: case_text_length, dtype: float64


## Step 4 — Define the delay-cause categories
Since we're not training the model on our own labels, we describe each category as a full sentence — this gives the model more context to match against than a single word would.

In [ ]:
# These are our candidate DELAY-CAUSE categories. Since this is "zero-shot"
# classification, the model was never specifically trained on these labels --
# instead, we describe each category in a natural sentence, and let the model
# judge which description is closest in meaning to each case's text.
#
# Writing each label as a short DESCRIPTIVE SENTENCE (not just a one-word tag)
# gives the model more context to match against, which improves accuracy.

delay_categories = {
    "Investigation pending":
        "The case is delayed because the police investigation is still ongoing or incomplete.",
    "Witness non-availability":
        "The case is delayed because witnesses were unavailable, absent, or retracted their statements.",
    "Repeated adjournments":
        "The case is delayed due to repeated adjournments or postponements of hearings.",
    "Procedural or documentation delay":
        "The case is delayed due to procedural issues, missing documents, or administrative backlog.",
    "No specific delay noted":
        "The case proceeded through the normal court process without any specific mention of delay.",
}

print("Delay-cause categories defined:")
for k in delay_categories:
    print(" -", k)


Delay-cause categories defined:
 - Investigation pending
 - Witness non-availability
 - Repeated adjournments
 - Procedural or documentation delay
 - No specific delay noted


## Step 5 — Load InLegalBERT

In [ ]:
# Load InLegalBERT -- a BERT model pretrained specifically on Indian legal text
# (as used in your seminar's proposed methodology and referenced in your
# literature survey: Paul et al., "Pre-trained Language Models for the Legal
# Domain: A Case Study on Indian Law").
#
# NOTE: the first time this runs, it will DOWNLOAD the model files from
# Hugging Face (a few hundred MB) -- make sure you have internet the first time.
# After that, it's cached locally and loads instantly.

MODEL_NAME = "law-ai/InLegalBERT"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)

# .eval() puts the model in "inference mode" -- we're only using it to read
# text and produce embeddings, not training/updating it, so this turns off
# training-only behaviors (like dropout) for more consistent results.
model.eval()

print("InLegalBERT loaded successfully")


e:\Projects\Judicial AI system with drift detection\.venv\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\91708\.cache\huggingface\hub\models--law-ai--InLegalBERT. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
e:\Projects\Judicial AI system with drift detection\.venv\Lib\site-packages\huggingface_hub\

InLegalBERT loaded successfully


## Step 6 — Build a function to turn any text into a numeric vector (embedding)
This is the core building block — both case texts and category descriptions will go through this same function, so they end up in the same numeric "space" and can be compared directly.

In [ ]:
# This function converts any piece of text into a single numeric vector
# (an "embedding") that captures its meaning. We'll use this on both the
# case texts AND the category descriptions, so we can compare them directly.

def get_embedding(text, max_length=512):
    # Tokenizer converts text into the numeric IDs the model understands.
    # truncation=True + max_length=512 -- InLegalBERT (like most BERT models)
    # can only read a limited amount of text at once, so long case texts
    # get cut off at 512 tokens (roughly 350-400 words).
    inputs = tokenizer(text, return_tensors="pt", truncation=True,
                        max_length=max_length, padding=True)

    # torch.no_grad() -- tells PyTorch we don't need to track gradients here,
    # since we're not training. This makes it faster and uses less memory.
    with torch.no_grad():
        outputs = model(**inputs)

    # outputs.last_hidden_state has one vector PER WORD/TOKEN in the text.
    # We take the MEAN across all tokens to get a single vector for the
    # whole text -- this is called "mean pooling", a standard way to turn
    # a BERT model's output into one sentence/document-level embedding.
    embedding = outputs.last_hidden_state.mean(dim=1)

    # Convert from a PyTorch tensor to a plain numpy array, and remove
    # the extra batch dimension (since we're only embedding one text at a time)
    return embedding.squeeze().numpy()

print("Embedding function ready")


Embedding function ready


## Step 7 — Embed the 5 category descriptions (done once)

In [ ]:
# Compute ONE embedding per delay-cause category description.
# We only need to do this once -- these don't change per case.

category_names = list(delay_categories.keys())
category_embeddings = np.array([
    get_embedding(delay_categories[name]) for name in category_names
])

print("Category embeddings computed. Shape:", category_embeddings.shape)
# Shape will be (5, 768) -- 5 categories, each represented by a 768-number vector
# (768 is InLegalBERT's embedding size, same as standard BERT-base)


Category embeddings computed. Shape: (5, 768)


## Step 8 — Embed every case's text
This is the slowest step — it runs the model once per case. Grab a coffee, or test on a handful of cases first using `df.head(10)` before running on all 365.

In [ ]:
# Now compute an embedding for EVERY case's combined text.
# NOTE: this loop calls the model once per case -- for 365 cases this can take
# a few minutes on a normal laptop CPU (no GPU needed, just patience).
# If you want to test quickly first, change df.iterrows() to df.head(10).iterrows()

case_embeddings = []
for idx, row in df.iterrows():
    emb = get_embedding(row['case_text'])
    case_embeddings.append(emb)
    if (idx + 1) % 50 == 0:
        print(f"Processed {idx + 1} / {len(df)} cases...")

case_embeddings = np.array(case_embeddings)
print("\nAll case embeddings computed. Shape:", case_embeddings.shape)


Processed 50 / 365 cases...
Processed 100 / 365 cases...
Processed 150 / 365 cases...
Processed 200 / 365 cases...
Processed 250 / 365 cases...
Processed 300 / 365 cases...
Processed 350 / 365 cases...

All case embeddings computed. Shape: (365, 768)


## Step 9 — Classify each case by comparing embeddings
Whichever category description is numerically closest (highest cosine similarity) to a case's text becomes that case's predicted delay cause.

In [ ]:
# For each case, compare its embedding to all 5 category embeddings using
# cosine similarity -- a measure of how "aligned" two vectors are
# (1.0 = identical meaning, 0 = unrelated, -1 = opposite).

raw_similarities = cosine_similarity(case_embeddings, category_embeddings)
# raw_similarities is (365, 5) -- one row per case, one column per category

# FIX -- raw InLegalBERT embeddings (mean-pooled, not fine-tuned for sentence
# similarity) suffer from "anisotropy": embeddings cluster in a narrow cone,
# so ONE category (the most generic-sounding one) ends up with a systematically
# higher baseline similarity to almost every case, regardless of content.
# Left uncorrected, argmax picks that same category almost every time -- which
# is exactly what was happening (361/365 cases -> "No specific delay noted").
#
# Fix: z-score each category's column ACROSS ALL CASES before comparing. This
# subtracts out each category's own baseline offset and leaves only the
# per-case RELATIVE preference -- which is where the real signal is.
col_mean = raw_similarities.mean(axis=0)
col_std = raw_similarities.std(axis=0)
similarities = (raw_similarities - col_mean) / col_std

predicted_indices = similarities.argmax(axis=1)          # which category won, per case
predicted_categories = [category_names[i] for i in predicted_indices]

# Report confidence using the ORIGINAL cosine score (still a meaningful raw
# number for the viva/demo) -- just don't use it to pick the category.
confidence_scores = raw_similarities[np.arange(len(df)), predicted_indices]

df['predicted_delay_cause'] = predicted_categories
df['delay_cause_confidence'] = confidence_scores

print("Delay-cause distribution across all cases:")
print(df['predicted_delay_cause'].value_counts())

# Save the calibration (per-category mean/std) so the live dashboard
# (app.py -> Predict New Case -> paste case text) can apply the SAME
# correction to a single new case, instead of falling back to raw,
# uncorrected cosine similarity.
import pickle
calibration = {
    'category_names': category_names,
    'col_mean': col_mean,
    'col_std': col_std,
}
with open(MODELS / "delay_cause_calibration.pkl", "wb") as f:
    pickle.dump(calibration, f)
print("Saved calibration -> models/delay_cause_calibration.pkl")


## Step 10 — Save the results
This becomes an input to Notebook 06 (priority scoring), where the delay cause will be one of the factors contributing to a case's urgency score.

In [ ]:
# Save the dataset with the new delay-cause predictions added --
# this file will feed into Notebook 06 (priority scoring).
df.to_csv(DATA / "dataset_with_delay_cause.csv", index=False)
print("Saved -> data/processed/dataset_with_delay_cause.csv", df.shape)


## Step 11 — Look at one case in detail (for your demo/viva)
This shows the similarity score against EVERY category, not just the winner — useful for explaining to your examiner exactly why the model picked what it picked.

In [ ]:
# ============================================================
# QUICK CHECK: classify ONE case by its case_id, and see the
# similarity score against EVERY category (not just the winner)
# -- useful for understanding WHY the model chose what it chose.
# Uses the SAME z-score-calibrated scores as Step 9, so the ranking
# shown here matches df['predicted_delay_cause'].
# ============================================================

case_id_to_check = 273  # <-- change this to any case_id

row = df[df['case_id'] == case_id_to_check].iloc[0]
text_embedding = get_embedding(row['case_text'])

raw_scores = cosine_similarity([text_embedding], category_embeddings)[0]
z_scores = (raw_scores - col_mean) / col_std

print(f"Case ID: {case_id_to_check}")
print(f"Case text (first 300 chars): {row['case_text'][:300]}...\n")
print("Category           raw cosine   z-score (used for the decision)")
for name, raw, z in sorted(zip(category_names, raw_scores, z_scores), key=lambda x: -x[2]):
    print(f"  {name:<35} {raw:.3f}          {z:+.2f}")

print(f"\n--> Predicted delay cause: {row['predicted_delay_cause']} "
      f"(raw cosine confidence: {row['delay_cause_confidence']:.3f})")
